In [1]:
# Setup variables
BRANCH = "dev"
import os
os.environ['MLFLOW_TRACKING_URI'] = "http://100.101.196.27:5000"
os.environ['COLAB_GPU'] = "True"


In [2]:
# Mount google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Clone repo
if not os.path.exists("/content/inat-phenology-cv"):
    !git clone -b {BRANCH} https://github.com/etiennegodin/inat-phenology-cv.git /content/inat-phenology-cv
%cd /content/inat-phenology-cv
!git pull origin {BRANCH} -q
!git restore .

Cloning into '/content/inat-phenology-cv'...
remote: Enumerating objects: 1589, done.
remote: Counting objects: 100% (744/744), done.
remote: Compressing objects: 100% (319/319), done.
remote: Total 1589 (delta 481), reused 609 (delta 352), pack-reused 845 (from 1)
Receiving objects: 100% (1589/1589), 289.20 KiB | 3.86 MiB/s, done.
Resolving deltas: 100% (1006/1006), done.
/content/inat-phenology-cv


In [4]:
%%capture
# Instal project requirements
!pip install -q --upgrade pip
!pip install -e . -q


In [5]:
%%capture
#Setup colab network to mlflow server
!curl -fsSL https://tailscale.com/install.sh | sh
!pip install -q "requests[socks]"

In [6]:
%%bash
sudo setsid nohup bash -c '
while true; do
  tailscaled \
    --tun=userspace-networking \
    --socks5-server=localhost:1055 \
    --state=/tmp/tailscale.state \
    >> /tmp/tailscaled.log 2>&1
  echo "[$(date)] tailscaled exited, restarting in 2s" >> /tmp/tailscaled.log
  sleep 2
done
' < /dev/null > /tmp/tailscaled_supervisor.log 2>&1 &
echo "supervisor launched"

supervisor launched


In [7]:
import subprocess
from google.colab import userdata
ts_auth_key = userdata.get("TAILSCALE_AUTH_KEY")
assert ts_auth_key
subprocess.run(
    [
        "sudo",
        "tailscale",
        "up",
        "--auth-key",
        ts_auth_key,
    ],
    check=True,
)
del ts_auth_key

TimeoutException: Requesting secret TAILSCALE_AUTH_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
!sudo tailscale status

100.79.154.107   5e9ea76b8795                       manateetiti@  linux    -                           
100.112.144.117  c318ba5607df                       manateetiti@  linux    offline, last seen 36m ago  
100.101.196.27   etienne-lenovo-ideapad-flex-15iml  manateetiti@  linux    -                           
100.112.113.49   pixel-7-pro                        manateetiti@  android  offline, last seen 18d ago  


In [ ]:
import requests

mlflow_proxies = {
    "http": "socks5h://localhost:1055",
    "https": "socks5h://localhost:1055",
}

r = requests.get(
    "http://100.101.196.27:5000/version",
    proxies=mlflow_proxies,
    timeout=5,
)

print(r.status_code)
print(r.text)

200
3.15.1


In [ ]:
# Copying mlflow.db from drive
os.makedirs("/content/data", exist_ok=True)
!cp -r "/content/drive/MyDrive/inat-phenology-cv/data/cv_photos3.parquet" "/content/data/"

In [ ]:
print("Copying images to local disk...")
if not os.path.exists("/content/images"):
  os.makedirs("/content/images", exist_ok=True)
  !tar -xf "/content/drive/MyDrive/inat-phenology-cv/data/images.tar.gz" -C /content/
  os.environ["INAT_IMAGE_DIR"] = "/content/images"

Copying images to local disk...


In [ ]:
!nvidia-smi

Sat Sep  5 19:44:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# Gated attention, no dropout
!torch_pipe train -n 30 --backbone bioclip --gated --attention_dropout 0.0 --seed 42 -name 'cv_inat_bioclip'
!torch_pipe train -n 30 --backbone bioclip2 --gated --attention_dropout 0.0 --seed 42 -name 'cv_inat_bioclip2'


INFO: Starting
Connecting to mlflow
Initalizing experiment
Running on cuda
open_clip_config.json: 100% 469/469 [00:00<00:00, 2.26MB/s]

open_clip_pytorch_model.bin: downloading bytes:  38% 229M/599M [00:01<00:01, 237MB/s, 16.6MB/s  ]  
open_clip_pytorch_model.bin: downloading bytes:  90% 541M/599M [00:02<00:00, 421MB/s, 46.2MB/s  ]
open_clip_pytorch_model.bin: reconstructing file:  90% 537M/599M [00:02<00:00, 229MB/s, 25.4MB/s  ]
open_clip_pytorch_model.bin: downloading bytes: 100% 567M/567M [00:02<00:00, 193MB/s, 50.3MB/s  ] ]
open_clip_pytorch_model.bin: reconstructing file: 100% 599M/599M [00:02<00:00, 204MB/s, 54.5MB/s  ]
Traceback (most recent call last):
  File "/usr/local/bin/torch_pipe", line 6, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/inat-phenology-cv/src/pytorch_pipeline/cli.py", line 437, in main
    exit_code = args.func(args, configs)
  File "/content/inat-phenology-cv/src/pytorch_pipeline/cli.py", line 110, in train_cmd
    datasets = build_d

In [ ]:
from google.colab import runtime
runtime.unassign()